# 누락된 값 (Missing Values) {#sec-missing-values}

## 서론

이 장에서는 누락된 값을 다루기 위한 도구와 요령을 살펴봅니다. 먼저 `NA`로 기록된 누락된 값을 처리하는 일반적인 도구들에 대해 논의하겠습니다. 그런 다음 데이터에서 단순히 존재하지 않는 값인 암시적 누락값(implicitly missing values)의 개념을 탐구하고, 이를 명시적으로 만드는 도구들을 보여드리겠습니다.
마지막으로 데이터에 나타나지 않는 카테고리로 인해 발생하는 비어 있는 그룹(empty groups)에 대한 논의로 마무리하겠습니다.

In [ ]:
# remove cell
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline

# Plot settings
plt.style.use("plot_style.txt")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

### 사전 준비

이 장에서는 **pandas** 데이터 분석 패키지를 사용합니다.

## 명시적 누락값 (Explicit Missing Values)

먼저 명시적 누락값, 즉 `NA`나 `nan`이 표시된 셀을 생성하거나 제거하는 유용한 도구들을 살펴보겠습니다.

### 누락된 값의 유형

**pandas**의 누락된 값들이 모두 똑같지는 않다는 점에 유의해야 합니다!

예를 들어, **pandas**의 실수(예: `float64` dtype)는 'nan'(Not a Number의 약자)을 사용합니다:

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame([5, 27.3, np.nan, -16], columns=["numbers"])
df

하지만 이것이 유일한 방법은 아닙니다! 파이썬 내장 `None` 값(여기서는 유효한 값들이 모두 부동 소수점이므로 NaN으로 변환됨)과 **pandas**의 `pd.NA`를 사용할 수도 있습니다:

In [ ]:
numbers = pd.DataFrame([pd.NA, 27.3, np.nan, -16, None], columns=["numbers"])
numbers

하지만 객체(object) 데이터 타입(문자열의 기본값)의 경우 이러한 타입들이 공존할 수 있습니다:

In [ ]:
fruits = pd.DataFrame(
    ["orange", np.nan, "apple", None, "banana", pd.NA], columns=["fruit"]
)
fruits

이러한 모든 유형의 누락된 값은 **pandas**의 `.isna()` 함수를 사용하여 찾을 수 있습니다. 이 함수는 값이 누락된 경우 `True`인 불리언 열을 반환합니다.

In [ ]:
fruits.isna()

편의상 `notna()` 함수도 제공됩니다.

### 명시적 누락값 처리하기

누락된 값을 처리하는 데는 여러 옵션이 있습니다. `fillna()` 함수가 이 역할을 합니다. 테스트 데이터를 사용하여 살펴보겠습니다:

In [ ]:
nan_df = pd.DataFrame(
    [
        [np.nan, 2, None, 0],
        [3, 4, np.nan, 1],
        [5, np.nan, np.nan, pd.NA],
        [np.nan, 3, np.nan, 4],
    ],
    columns=list("ABCD"),
)

nan_df

먼저, 모든 누락된 값을 단일 고정값으로 채울 수 있습니다:

In [ ]:
nan_df.fillna(0)

열 단위로 수행할 수도 있습니다. 여기서는 'A', 'B', 'C', 'D' 열의 모든 NaN 요소를 각각 0, 1, 2, 3으로 바꿉니다.

In [ ]:
nan_df.fillna(value={"A": 0, "B": 1, "C": 2, "D": 3})

또한 null이 아닌 값을 (인덱스 기준으로) 앞으로 또는 뒤로 전파할 수 있습니다.

In [ ]:
nan_df.fillna(method="ffill")

In [ ]:
nan_df.fillna(method="bfill")

전방 채우기(forward fill)와 후방 채우기(backward fill) 옵션은 시계열 데이터에서 특히 유용합니다 - 단, 예측 모델링을 수행 중이라면 주의해서 사용해야 합니다!

이러한 모든 함수의 또 다른 특징은 `limit=` 키워드 인수를 사용하여 대체되는 NaN의 개수를 제한할 수 있다는 점입니다.

In [ ]:
nan_df.fillna(value={"A": 0, "B": 1, "C": 2, "D": 3}, limit=1)

물론 다른 옵션은 누락된 값을 완전히 필터링하는 것입니다. 전체 행(`axis=0`)을 제거할지 아니면 열(`axis=1`)을 제거할지에 따라 몇 가지 방법이 있습니다. (단, 이 경우 각 열에 최소 하나의 NaN이 있으므로 열 기준 제거 시 데이터가 남지 않을 것입니다!)

In [ ]:
nan_df["A"].dropna(axis=0)  # 단일 열에서 수행

In [ ]:
nan_df.dropna(axis=1)

`dropna()`도 몇 가지 키워드 인수를 받습니다. 예를 들어 `how="all"`은 행이나 열의 *모든* 값이 NA인 경우에만 삭제합니다.

In [ ]:
nan_df.dropna(how="all")

`thresh`(임계값) 키워드도 있습니다 - 이를 사용하면 최소 몇 개 이상의 누락되지 않은 관측치가 있는 행이나 열만 유지할 수 있습니다.

NaN을 필터링하는 또 다른 방법은 불리언 열을 사용하는 일반적인 필터링 방법을 `.notna()` 함수와 조합하여 사용하는 것입니다. 아래 예제에서 'A'가 NA가 아닌 행들에 대해 모든 열을 확인합니다.

In [ ]:
nan_df[nan_df["A"].notna()]

### NA 값 추가하기

가끔은 반대로 어떤 구체적인 값이 실제로는 누락된 값을 나타내는 문제를 겪을 수도 있습니다. 이는 대개 누락된 값을 표현하는 적절한 방법이 없는 오래된 소프트웨어에서 생성된 데이터에서 발생하며, 99나 -999와 같은 특별한 값을 대신 사용합니다.

가능하다면 데이터를 읽어올 때 이를 처리하세요. 예를 들어 `pd.read_csv()`를 호출할 때 `na_values=` 키워드 인수를 사용하는 것입니다. 나중에 문제를 발견했거나 데이터 소스에서 읽기 시 처리를 제공하지 않는다면, 제공된 데이터를 교체하기 위해 다양한 옵션을 사용할 수 있습니다:

In [ ]:
stata_df = pd.DataFrame([[3, 4, 5], [-7, 4, -99], [-99, 6, 5]], columns=list("ABC"))

stata_df

가장 쉬운 옵션은 아마도 `.replace()`일 것입니다:

In [ ]:
stata_df.replace({-99: pd.NA})

`.replace()`는 딕셔너리를 받기 때문에 여러 값을 한 번에 교체할 수 있습니다:

In [ ]:
stata_df.replace({-99: pd.NA, -7: pd.NA})

이는 데이터 프레임의 *모든* 열에 적용된다는 점에 유의하세요. 단 하나에만 적용하려면 특정 열을 먼저 선택하세요.

## 암시적 누락값 (Implicit Missing Values)

지금까지는 데이터에서 확인할 수 있는 `NA`와 같은 **명시적** 누락값에 대해 이야기했습니다.
하지만 누락된 값은 전체 데이터 행 자체가 데이터에서 단순히 누락된 경우인 **암시적** 누락값이 될 수도 있습니다.
매 분기 주식 가격을 기록하는 간단한 데이터셋으로 차이를 설명해 보겠습니다:

In [ ]:
stocks = pd.DataFrame(
    {
        "year": [2020, 2020, 2020, 2020, 2021, 2021, 2021],
        "qtr": [1, 2, 3, 4, 2, 3, 4],
        "price": [1.88, 0.59, 0.35, np.nan, 0.92, 0.17, 2.66],
    }
)
stocks

이 데이터셋에는 두 개의 누락된 관측치가 있습니다:

-   2020년 4분기 가격은 값이 `NA`이므로 명시적으로 누락되었습니다.

-   2021년 1분기 가격은 데이터셋에 아예 나타나지 않으므로 암시적으로 누락되었습니다.

차이를 생각하는 한 가지 방법은 다음과 같은 화두(koan)를 통하는 것입니다:

> 명시적 누락값은 부재(absence)의 존재(presence)이다.
>
> 암시적 누락값은 존재(presence)의 부재(absence)이다.

가끔은 물리적으로 다룰 수 있도록 암시적 누락값을 명시적으로 만들고 싶을 때가 있습니다.
다른 경우에는 데이터 구조에 의해 강제된 명시적 누락값을 제거하고 싶을 수도 있습니다.
다음 섹션들에서는 암시적 누락과 명시적 누락 사이를 이동하는 몇 가지 도구들에 대해 논의합니다.

### 피벗하기 (Pivoting)

암시적 누락값을 명시적으로 만들 수 있는 도구 하나를 이미 보았습니다: 피벗(pivoting)입니다. 데이터를 넓게 만들면 모든 행과 새로운 열의 조합에 어떤 값이 있어야 하므로 암시적 누락값이 명시적이 될 수 있습니다. 예를 들어 `stocks`를 피벗하여 `quarter`를 열로 두면(그리고 `year`를 인덱스로 만들면), 두 누락된 값 모두 명시적이 됩니다:

In [ ]:
stocks.pivot(columns="qtr", values="price", index="year")

기본적으로 데이터를 길게 만들면 명시적 누락값은 보존됩니다.

### 범주형 변수의 누락된 값

누락된 값의 마지막 유형은 비어 있는 그룹(empty group)으로, 범주형 데이터를 다룰 때 발생할 수 있는 데이터에 나타나지 않는 그룹입니다.

예를 들어 사람들에 대한 건강 정보가 담긴 데이터셋이 있다고 가정해 봅시다:

In [ ]:
health = pd.DataFrame(
    {
        "name": ["Ikaia", "Oletta", "Leriah", "Dashay", "Tresaun"],
        "smoker": ["no", "no", "previously", "no", "yes"],
        "age": [34, 88, 75, 47, 56],
    }
)
health["smoker"] = health["smoker"].astype("category")

이제 마지막 행의 데이터를 삭제합니다:

In [ ]:
health_cut = health.iloc[:-1, :]
health_cut

이제 smoker의 'yes' 값은 우리 데이터 프레임의 어디에도 나타나지 않는 것처럼 보입니다. 하지만 카테고리별 빈도수를 얻기 위해 `value_counts()`를 실행하면, 데이터 프레임은 현재 존재하지 않는 'yes' 카테고리가 있다는 것을 '기억'하고 있음을 볼 수 있습니다:

In [ ]:
health_cut["smoker"].value_counts()

`groupby()` 연산에서도 같은 현상이 일어나는 것을 볼 수 있습니다:

In [ ]:
health_cut.groupby("smoker")["age"].mean()

존재하지 않는 숫자의 평균을 구했기 때문에 yes 행에 실제 값 대신 NaN을 얻게 된 것을 볼 수 있습니다(하지만 'yes' 행 자체는 존재합니다).